**05 - Opinion Insights**

En este notebook se extraen los principales aspectos positivos y negativos presentes en las opiniones de los clientes.

El objetivo es complementar el modelo de clasificación de sentimiento proporcionando información útil para empresas de ecommerce sobre los temas más mencionados por los usuarios.

In [1]:
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer

In [2]:
from google.colab import files

uploaded = files.upload()

Saving amazon_reviews_clean.csv to amazon_reviews_clean.csv


In [5]:
df = pd.read_csv("amazon_reviews_clean.csv")

In [4]:
df = pd.read_csv("amazon_reviews_clean.csv")
df.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,review,review_length,review_clean,sentiment
0,3.0,Smells like gasoline! Going back!,First & most offensive: they reek of gasoline ...,[{'small_image_url': 'https://m.media-amazon.c...,B083NRGZMM,B083NRGZMM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1658185117948,0,True,Smells like gasoline! Going back! First & most...,1467,smell like gasoline going back first offensive...,Neutro
1,1.0,Didn’t work at all lenses loose/broken.,These didn’t work. Idk if they were damaged in...,[],B07N69T6TM,B07N69T6TM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1592678549731,0,True,Didn’t work at all lenses loose/broken. These ...,265,didnt work lens loosebroken didnt work idk dam...,Negativo
2,5.0,Excellent!,I love these. They even come with a carry case...,[],B01G8JO5F2,B01G8JO5F2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1523093017534,0,True,Excellent! I love these. They even come with a...,480,excellent love even come carry case several si...,Positivo
3,5.0,Great laptop backpack!,I was searching for a sturdy backpack for scho...,[],B001OC5JKY,B001OC5JKY,AGGZ357AO26RQZVRLGU4D4N52DZQ,1290278495000,18,True,Great laptop backpack! I was searching for a s...,1112,great laptop backpack searching sturdy backpac...,Positivo
4,5.0,Best Headphones in the Fifties price range!,I've bought these headphones three times becau...,[],B013J7WUGC,B07CJYMRWM,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,1676601581238,0,True,Best Headphones in the Fifties price range! I'...,284,best headphone fifty price range ive bought he...,Positivo


**Separación de reviews por sentimiento**

Se separan las opiniones positivas y negativas para poder extraer, por separado, los aspectos mejor y peor valorados de cada producto.

In [13]:
#separar por sentimiento

df = df.dropna(subset=["review_clean"])

positivas = df[df["sentiment"] == "Positivo"]

negativas = df[df["sentiment"] == "Negativo"]

print("Reviews positivas:", len(positivas))
print("Reviews negativas:", len(negativas))

Reviews positivas: 40277
Reviews negativas: 6259


**Selección de producto**

El análisis se realiza a nivel de producto utilizando `parent_asin` como identificador. Se selecciona el producto con más reviews para probar el funcionamiento del sistema.

In [27]:
#agrupar por producto
df.groupby("parent_asin")

In [30]:
#revisamos cuantos productos hay en el df
print("Número de productos:", df["parent_asin"].nunique())
df["parent_asin"].value_counts().head(10)

Número de productos: 34322


,count
parent_asin,
B075X8471B,166
B01K8B8YA8,143
B07GZFM1ZM,120
B0791TX5P5,103
B07H65KP63,103
B010BWYDYA,98
B07S764D9V,85
B07KTYJ769,77
B0BW4PFM58,55


In [32]:
producto = df["parent_asin"].value_counts().index[0]

print(producto)

producto_df = df[df["parent_asin"] == producto]

producto_df.head()

B075X8471B


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,review,review_length,review_clean,sentiment
19,4.0,Four Stars,Pretty cool!!!<br /><br />Thanks Amazon.,[],B00ZV9RDKK,B075X8471B,AEM663T6XHZFWLODF4US2RCOCUSA,1509491262120,0,True,Four Stars Pretty cool!!!<br /><br />Thanks Am...,51,four star pretty coolbr br thanks amazon,Positivo
248,5.0,Five Stars,We love it,[],B00ZV9RDKK,B075X8471B,AEJDETWITK2KGACH7FUBMY33PPSQ,1519939757127,0,True,Five Stars We love it,21,five star love,Positivo
383,5.0,AWESOME and simple to set up,i have hemmed and hawed for quite a while abo...,[],B00ZV9RDKK,B075X8471B,AHGAOIZVODNHYMNCBV4DECZH42UQ,1532476996921,0,True,AWESOME and simple to set up i have hemmed and...,862,awesome simple set hemmed hawed quite fire sti...,Positivo
2113,4.0,Fire Stick Works great but can interfere with ...,Oct 2017 Review Update: Amazon has done a gre...,[],B00ZV9RDKK,B075X8471B,AEDHXXX2F66EMWOSSPCQHS62UKNQ,1482384682000,1,True,Fire Stick Works great but can interfere with ...,313,fire stick work great interfere tv antenna you...,Positivo
2156,5.0,Would absolutely buy again for another tv,Love love love. Used to have Google Chromecast...,[],B00ZV9RDKK,B075X8471B,AH2CJIKJPIIRDG264ZWX2UUQN5SQ,1522244290750,0,True,Would absolutely buy again for another tv Love...,136,would absolutely buy another tv love love love...,Positivo


In [33]:
print("Número de reviews:", len(producto_df))
producto_df["sentiment"].value_counts()

Número de reviews: 166


,count
sentiment,
Positivo,141
Negativo,17
Neutro,8


In [34]:
producto_positivas = producto_df[
    producto_df["sentiment"] == "Positivo"
]

producto_negativas = producto_df[
    producto_df["sentiment"] == "Negativo"
]

print("Positivas:", len(producto_positivas))
print("Negativas:", len(producto_negativas))

Positivas: 141
Negativas: 17


**Primera aproximación: palabras frecuentes**

Se utiliza CountVectorizer para extraer las palabras más frecuentes en las reviews positivas y negativas. Esta primera aproximación permite detectar términos relevantes, aunque puede incluir palabras demasiado genéricas.

In [35]:
from sklearn.feature_extraction.text import CountVectorizer

stopwords_extra = [
    "product", "good", "great", "love", "like", "work",
    "using", "use", "used", "buy", "bought", "amazon",
    "one", "would", "get", "got", "really", "well",
    "br", "im", "ive", "dont", "didnt", "doesnt"
]

def obtener_aspectos(textos, n=5):

    vectorizador = CountVectorizer(
        stop_words="english",
        max_features=1000
    )

    matriz = vectorizador.fit_transform(textos.dropna())

    palabras = vectorizador.get_feature_names_out()
    frecuencias = matriz.sum(axis=0).A1

    resultado = pd.DataFrame({
        "palabra": palabras,
        "frecuencia": frecuencias
    })

    resultado = resultado.sort_values(
        by="frecuencia",
        ascending=False
    )

    resultado = resultado[
        ~resultado["palabra"].isin(stopwords_extra)
    ]

    return resultado.head(n)

In [37]:
#llamar a la función
aspectos_positivos = obtener_aspectos(
    producto_positivas["review_clean"],
    n=5
)

aspectos_negativos = obtener_aspectos(
    producto_negativas["review_clean"],
    n=5
)


aspectos_negativos
aspectos_positivos

,palabra,frecuencia
602,tv,67
556,stick,62
166,easy,36
554,star,34
73,cable,21


In [38]:
print("Producto:", producto)
print("Total reviews:", len(producto_df))

print("\nDistribución de sentimiento:")
print(producto_df["sentiment"].value_counts(normalize=True).round(2) * 100)

print("\nAspectos positivos:")
print(", ".join(aspectos_positivos["palabra"].tolist()))

print("\nAspectos negativos:")
print(", ".join(aspectos_negativos["palabra"].tolist()))

Producto: B075X8471B
Total reviews: 166

Distribución de sentimiento:
sentiment
Positivo    85.0
Negativo    10.0
Neutro       5.0
Name: proportion, dtype: float64

Aspectos positivos:
tv, stick, easy, star, cable

Aspectos negativos:
remote, tv, stick, app, free


Ahora voy a instalar spaCy. mejora la extracción de los aspectos del producto.

**Extracción de aspectos con spaCy**

Para mejorar la extracción de aspectos, se utiliza spaCy con el objetivo de identificar principalmente sustantivos, ya que suelen representar componentes, características o elementos del producto mencionados por los usuarios.

In [40]:
!pip install spacy -q
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 49.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [41]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [42]:
def extraer_sustantivos(texto):

    doc = nlp(texto)

    sustantivos = []

    for token in doc:
        if token.pos_ in ["NOUN", "PROPN"]:
            sustantivos.append(token.lemma_.lower())

    return sustantivos

In [43]:
texto_prueba = producto_df["review"].iloc[0]

print(texto_prueba)

print(extraer_sustantivos(texto_prueba))

Four Stars Pretty cool!!!<br /><br />Thanks Amazon.
['star', 'pretty', 'amazon']


In [46]:
from collections import Counter

palabras_excluir = [
    "amazon", "product", "thing", "time", "day", "month", "year",
    "star", "stars", "tv", "stick", "fire", "br", "show"
]

def obtener_aspectos_spacy(textos, n=5):

    todos_aspectos = []

    for texto in textos.dropna():
        aspectos = extraer_sustantivos(texto)
        todos_aspectos.extend(aspectos)

    contador = Counter(todos_aspectos)

    for palabra in palabras_excluir:
        contador.pop(palabra, None)

    aspectos_frecuentes = contador.most_common(n)

    return aspectos_frecuentes

In [47]:
aspectos_positivos_spacy = obtener_aspectos_spacy(
    producto_positivas["review"],
    n=5
)

aspectos_negativos_spacy = obtener_aspectos_spacy(
    producto_negativas["review"],
    n=5
)

print("Aspectos positivos:", aspectos_positivos_spacy)
print("Aspectos negativos:", aspectos_negativos_spacy)

Aspectos positivos: [('cable', 22), ('device', 18), ('netflix', 16), ('voice', 14), ('love', 13)]
Aspectos negativos: [('remote', 9), ('app', 9), ('internet', 5), ('alexa', 4), ('channel', 4)]



En este notebook se ha desarrollado un primer sistema de extracción de insights por producto.

A partir de las reviews de un producto concreto, se calcula la distribución de sentimientos y se extraen los aspectos más mencionados en las opiniones positivas y negativas.

La primera aproximación con CountVectorizer permite identificar palabras frecuentes, aunque incluye términos genéricos. Por ello, se incorpora spaCy para mejorar la extracción de aspectos centrándose en sustantivos relacionados con características del producto.

Este enfoque será la base para la futura aplicación en Streamlit, donde el usuario podrá subir un CSV de reviews y obtener un resumen automático por producto.